In [ ]:
from typing_extensions import TypedDict
from rich import print


class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: int


movie: MovieDict = {
    "title1": "インセプション",
    "year": 2010,
    "director": "クリストファー・ノーラン",
    "rating": 8.8,
}

print(movie)


In [15]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .envファイルから環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [16]:
from typing_extensions import TypedDict, Annotated


class MovieTypedDict(TypedDict):
    """
    映画の詳細情報
    """
    title: Annotated[str, "映画の正式名称、例：『インセプション』"]
    year: Annotated[int, "映画の公開年、4桁の数字で表す"]
    director: Annotated[str, "映画監督のフルネーム"]
    rating: Annotated[float, "映画の10点満点での評価、小数点第1位まで含めることができる"]


# モデルの構造化出力を設定
structured_llm = model.with_structured_output(MovieTypedDict)
# モデルを呼び出し、構造化出力を取得
response = structured_llm.invoke("映画『インターステラー』を紹介してください")
print(type(response))
print(response)


<class 'dict'>

{'title': 'インターステラー', 'year': 2014, 'director': 'クリストファー・ノーラン', 'rating': 8.6}

ネスト形式

In [17]:
from typing import TypedDict, List, Annotated


# TypedDict を使ってネスト構造を定義
class Actor(TypedDict):
    """俳優情報"""
    name: Annotated[str, "俳優名"]
    role: Annotated[str, "演じる役柄"]


class Movie(TypedDict):
    """映画情報"""
    title: Annotated[str, "映画タイトル"]
    year: Annotated[int, "公開年"]
    director: Annotated[str, "監督"]
    cast: Annotated[List[Actor], "俳優リスト"]  # ネストされたリストを定義
    rating: Annotated[float, "評価"]


# モデルの構造化出力を設定
structured_llm = model.with_structured_output(Movie)
# モデルを呼び出し、構造化出力を取得
resp = structured_llm.invoke("映画『インセプション』を紹介してください")
print(resp)


{
    'title': 'インセプション',
    'year': 2010,
    'director': 'クリストファー・ノーラン',
    'cast': [
        {'name': 'レオナルド・ディカプリオ', 'role': 'ドム・コブ'},
        {'name': 'ジョセフ・ゴードン＝レヴィット', 'role': 'アーサー'},
        {'name': 'エレン・ペイジ', 'role': 'アリアドネ'},
        {'name': 'トム・ハーディ', 'role': 'イームス'},
        {'name': 'ケン・ワタナベ', 'role': 'サイファー'},
        {'name': 'マリオン・コティヤール', 'role': 'モル'}
    ],
    'rating': 8.8
}

...必須フィールド

In [18]:
class MovieDict(TypedDict):
    """
    映画の詳細情報
    """
    title: Annotated[str, ..., "映画の正式名称、例：『インセプション』"]
    year: Annotated[int, ..., "映画の公開年、4桁の数字で表す"]
    director: Annotated[str, ..., "映画監督のフルネーム"]
    rating: Annotated[float, ..., "映画の10点満点での評価、小数点第1位まで含めることができる"]
    
# モデルの構造化出力を設定
structured_llm = model.with_structured_output(Movie)
# モデルを呼び出し、構造化出力を取得
resp = structured_llm.invoke("映画『インセプション』を紹介してください")
print(resp)

{
    'title': 'インセプション',
    'year': 2010,
    'director': 'クリストファー・ノーラン',
    'cast': [
        {'name': 'レオナルド・ディカプリオ', 'role': 'ドム・コブ'},
        {'name': 'ジョセフ・ゴードン＝レヴィット', 'role': 'アーサー'},
        {'name': 'エレン・ペイジ', 'role': 'アリアドネ'},
        {'name': 'トム・ハーディ', 'role': 'EAMES'},
        {'name': 'ケン・ワタナベ', 'role': 'サウスウィック'},
        {'name': 'マリオン・コティヤール', 'role': 'モル'}
    ],
    'rating': 8.8
}